# 9b. Sea ice: figures

Everything here is drawn from what notebook 09a saved, so nothing slow runs. 09a explains each step and has a quick look at each.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from extant import paths, sea_ice
from extant.logger import setup_logging
from extant.plots import era5_evaluation as pe, forced_response as fr, paper_panels, sea_ice as sip

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

## Settings

In [ ]:
#(t): The experiments compared with hist-nat, in the order of every figure, and the seasons, the columns of every map
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The model and experiment of the one-model figures
MODEL = 'CanESM5'
EXPERIMENT = 'hist-GHG'

#(t): As in 09a: the final years compared with hist-nat, and the years compared with ERA5
WINDOW = 21
ERA5_YEARS = slice(1979, 2013)

### Colour scales

One per quantity. A difference in concentration is blue where there is more ice and red where there is less, so that ice lost reads like warming on the temperature maps.

In [ ]:
concentration_scale = fr.Scale(np.arange(0, 101, 10), 'Blues', 'Sea-ice concentration (%)', extend='neither')
concentration_change_scale = fr.Scale(np.arange(-55, 56, 10), 'RdBu', 'Difference in sea-ice concentration (%)',
                                      ticks=np.arange(-50, 51, 10))
correlation_scale = fr.Scale(np.round(np.arange(-1, 1.01, 0.2), 1), 'PuOr_r', 'Correlation of sea ice and temperature',
                             extend='neither')
zone_scale = fr.class_scale(sea_ice.ZONES, ['#f4f3ef', '#2c7fb8', '#e6550d', '#31a354', '#bdbdbd'], 'Zone')
tas_scales = fr.colour_scales('tas')

## Which results

Run **one** of the next two sections: they say whether to open the sample's results or the full data's.

### Sample data

In [ ]:
RUN = 'sample'

### Full data (JASMIN)

In [ ]:
RUN = 'full'

## Open the results

First where each one is (`paths.find`: DATA_DIR first, then SCRATCH), then each opened.

In [ ]:
#(t): The results of 09a used here
RESULTS = ['regrid_example', 'extent', 'climatology', 'ocean_area', 'era5_extent', 'historical_cycle', 'era5_cycle',
           'historical_climatology', 'era5_climatology', 'historical_member_edges', 'era5_edge', 'ice_change',
           'extent_change', 'zones', 'zone_means', 'ice_region_change']
result_paths = {name: paths.find(paths.result(name, 'siconc', RUN)) for name in RESULTS}
result_paths

In [ ]:
#(t): Section 1's example: one field on the model's own grid and on the 2.5° grid
regrid_example_ds = xr.open_zarr(result_paths['regrid_example']).load()
regrid_example_ds

In [ ]:
#(t): Each member's seasonal extent and its model's continent (km²), /<model>/<experiment>
ice_extents_tree = xr.open_datatree(result_paths['extent'], engine='zarr').load()
ice_extents_tree

In [ ]:
#(t): The ensemble-mean concentration over each experiment's final WINDOW years (%)
ice_climatology_da = xr.open_zarr(result_paths['climatology'])['siconc'].load()
ice_climatology_da

In [ ]:
#(t): Each cell's ocean (km²), every model and experiment
ocean_area_da = xr.open_zarr(result_paths['ocean_area'])['ocean_area'].load()

In [ ]:
#(t): Each historical member's seasonal cycle of extent over ERA5_YEARS (million km²), /<model>/historical, and ERA5's
historical_cycle_tree = xr.open_datatree(result_paths['historical_cycle'], engine='zarr').load()
era5_cycle_da = xr.open_zarr(result_paths['era5_cycle'])['extent'].load()

In [ ]:
#(t): Each model's historical concentration over ERA5_YEARS, and ERA5's (%)
historical_climatology_da = xr.open_zarr(result_paths['historical_climatology'])['siconc'].load()
era5_climatology_da = xr.open_zarr(result_paths['era5_climatology'])['siconc'].load()

In [ ]:
#(t): Each historical member's ice edge over ERA5_YEARS (degrees), /<model>/historical, and ERA5's
historical_member_edges_tree = xr.open_datatree(result_paths['historical_member_edges'], engine='zarr').load()
era5_edge_da = xr.open_zarr(result_paths['era5_edge'])['edge_latitude'].load()

In [ ]:
#(t): The change in concentration (%), extent (million km²) and edge (degrees) against hist-nat
ice_change_da = xr.open_zarr(result_paths['ice_change'])['siconc'].load()
extent_change_ds = xr.open_zarr(result_paths['extent_change']).load()

In [ ]:
#(t): Each cell's zone, the temperature changes in each zone, and over hist-nat's ice
ice_zones_da = xr.open_zarr(result_paths['zones'])['zone'].load()
tas_zone_means_ds = xr.open_zarr(result_paths['zone_means']).load()
ice_region_change_ds = xr.open_zarr(result_paths['ice_region_change']).load()

In [ ]:
#(t): Notebook 03's mean and width changes, for the figures of what goes into the zones
mean_change_da = xr.open_zarr(paths.find(paths.result('mean_bootstrap', 'tas', RUN)))['mean_change'].load()
width_change_da = xr.open_zarr(paths.find(paths.result('qrange_change', 'tas', RUN)))['qrange_change'].load()

## 1. How a field goes onto the 2.5° grid

09a, section 1, on one month of one sample file.

In [ ]:
fig, axes = sip.regrid_schematic(regrid_example_ds['native'], regrid_example_ds['native_lat'],
                                 regrid_example_ds['native_lon'], regrid_example_ds['on_grid'],
                                 title=f'How a field goes onto the 2.5° grid: {MODEL}, historical, September 2013')

## 2. The extent and the ice edge

**The ice edge as a latitude**, on `MODEL`'s hist-nat ensemble mean in JJA: the concentration and its 15% line; the cells that count (ice, and the continent), with the circle that encloses the same area; and the cap's area read back as a latitude.

In [ ]:
#(t): The field, and its cells' ocean
example_field_da = ice_climatology_da.sel(model=MODEL, experiment='hist-nat', season='JJA')
example_ocean_area_da = ocean_area_da.sel(model=MODEL, experiment='hist-nat')

In [ ]:
#(t): What the figure draws: the ice cells, the continent, the extent and the edge (sea_ice.edge_example)
example_edge_ds = sea_ice.edge_example(example_field_da, ocean_area=example_ocean_area_da)
example_edge_ds

In [ ]:
fig, axes = sip.edge_schematic(example_edge_ds, title=f'How the ice edge becomes a latitude: {MODEL}, hist-nat, '
                                                      f'final {WINDOW} years, JJA')

**Each experiment's extent through time**: the ensemble mean and the members' 5–95%.

In [ ]:
#(t): The ensemble mean and the 5% and 95% of the members, every year (sea_ice.extent_summary)
ice_extent_summary_ds = sea_ice.extent_summary(ice_extents_tree)
ice_extent_summary_ds

In [ ]:
fig, axes = sip.extent_timeseries(ice_extent_summary_ds, seasons=SEASONS,
                                  title='Sea-ice extent, every model, experiment and season')

## 3. Against ERA5: is the models' sea ice realistic?

Each model's historical members against ERA5 over `ERA5_YEARS`, counted on the same grid in the same way. The usual CMIP6 picture (Roach et al., 2020): most models have too little ice in summer and a spread of winter extents round the observed one.

### The seasonal cycle

In [ ]:
fig, axes = sip.seasonal_cycle(historical_cycle_tree, era5_cycle_da,
                               title=f'Seasonal cycle of sea-ice extent, historical against ERA5, '
                                     f'{ERA5_YEARS.start}-{ERA5_YEARS.stop}')

### Where the edge is

The historical ensemble-mean concentration minus ERA5's, with both edges: the model's (black) and ERA5's (orange). Blue: the model has more ice.

In [ ]:
#(t): The difference
historical_bias_da = historical_climatology_da - era5_climatology_da

In [ ]:
#(t): The two edges drawn on each map: the model's, and ERA5's (the same on every row)
model_era5_edges_da = xr.concat([historical_climatology_da, era5_climatology_da.broadcast_like(historical_climatology_da)],
                                dim=pd.Index(['model', 'ERA5'], name='edge'))

In [ ]:
panels = fr.map_grid(historical_bias_da.sel(season=SEASONS), concentration_change_scale, row_dim='model',
                     col_dim='season', edges=model_era5_edges_da.sel(season=SEASONS),
                     title=f'Sea-ice concentration, historical minus ERA5, {ERA5_YEARS.start}-{ERA5_YEARS.stop}')

### The edge as a latitude

In [ ]:
fig, axes = sip.edge_latitudes(historical_member_edges_tree, era5_edge_da.sel(season=SEASONS),
                               title=f'Where the ice edge is: historical against ERA5, {ERA5_YEARS.start}-{ERA5_YEARS.stop}')

### The extent's mean, variability and trend: ERA5 among the members (JASMIN)

09a, section 6. A tick: ERA5 inside the members' 5–95%; a cross: outside (and what that means for the model); a dash: too few members to flag anything.

In [ ]:
#(t): ERA5 among each model's members, /<model>/historical, on (statistic, season, member)
era5_extent_test_tree = xr.open_datatree(paths.find(paths.result('era5_extent_test', 'siconc', RUN)), engine='zarr').load()
era5_extent_test_tree

In [ ]:
fig, axes = sip.era5_strips(era5_extent_test_tree, 'mean')

In [ ]:
fig, axes = sip.era5_strips(era5_extent_test_tree, 'std')

In [ ]:
fig, axes = sip.era5_strips(era5_extent_test_tree, 'trend')

In [ ]:
#(t): Every model's verdicts, on (model, statistic, season)
era5_extent_verdicts_ds = xr.concat([node.to_dataset()[['percentile', 'verdict', 'testable']]
                                     for node in era5_extent_test_tree.leaves],
                                    dim=xr.Variable('model', [node.parent.name for node in era5_extent_test_tree.leaves]))
era5_extent_verdicts_ds

In [ ]:
panels = pe.scorecard(era5_extent_verdicts_ds.sel(season=SEASONS),
                      title=f"Sea-ice extent: where ERA5 falls among each model's historical members, "
                            f"{ERA5_YEARS.start}-{ERA5_YEARS.stop}")

## 4. Sea ice and the temperature changes

### The change in sea ice

Each experiment's ensemble-mean concentration over the final `WINDOW` years minus hist-nat's, with hist-nat's edge (dashed) and the experiment's (solid). Blue: more ice; red: less.

In [ ]:
#(t): MODEL's concentration, and its two edges for EXPERIMENT
model_ice_da = ice_climatology_da.sel(model=MODEL, season=SEASONS)
model_edges_da = sip.edge_pair(model_ice_da, EXPERIMENT)

In [ ]:
#(t): What goes in: hist-nat, EXPERIMENT, and the change
panels = fr.row_grid(
    [fr.Row(model_ice_da.sel(experiment='hist-nat'), concentration_scale, f'hist-nat\nfinal {WINDOW} years',
            edges=model_edges_da.sel(edge=['hist-nat'])),
     fr.Row(model_ice_da.sel(experiment=EXPERIMENT), concentration_scale, f'{EXPERIMENT}\nfinal {WINDOW} years',
            edges=model_edges_da.sel(edge=['experiment'])),
     fr.Row(ice_change_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS), concentration_change_scale,
            f'Change\n{EXPERIMENT} − hist-nat', edges=model_edges_da)],
    title=f'What goes into the change in sea ice: {MODEL}, {EXPERIMENT}',
)

In [ ]:
#(t): One model, every experiment
panels = fr.map_grid(ice_change_da.sel(model=MODEL, season=SEASONS), concentration_change_scale,
                     row_dim='experiment', col_dim='season', edges=sip.edge_pair(model_ice_da, EXPERIMENTS),
                     title=f'{MODEL}: change in sea-ice concentration, final {WINDOW} years against hist-nat')

In [ ]:
#(t): Every model, one figure per experiment
for experiment in EXPERIMENTS:
    panels = fr.map_grid(ice_change_da.sel(experiment=experiment, season=SEASONS), concentration_change_scale,
                         row_dim='model', col_dim='season',
                         edges=sip.edge_pair(ice_climatology_da.sel(season=SEASONS), experiment),
                         title=f'Change in sea-ice concentration, final {WINDOW} years against hist-nat: {experiment}')

In [ ]:
#(t): The multi-model median of the change, with the edges of the median concentration
median_ice_da = ice_climatology_da.sel(season=SEASONS).median('model')
panels = fr.map_grid(ice_change_da.sel(season=SEASONS).median('model'), concentration_change_scale,
                     row_dim='experiment', col_dim='season', edges=sip.edge_pair(median_ice_da, EXPERIMENTS),
                     title=f'Change in sea-ice concentration, final {WINDOW} years against hist-nat, multi-model median')

In [ ]:
#(t): The change in extent (million km²) and in the edge's latitude (degrees; negative: further south), every model
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
extent_change_ds['extent_change'].sel(season=SEASONS).mean('season').plot.line(ax=axes[0], x='experiment', hue='model',
                                                                                 marker='o')
extent_change_ds['edge_change'].sel(season=SEASONS).mean('season').plot.line(ax=axes[1], x='experiment', hue='model',
                                                                               marker='o');

### Where the ice went: the temperature changes in each zone

Each ocean cell sorted by its concentration in hist-nat's final years and the experiment's: open water in both, ice in both, ice lost, ice gained, or the continent. If the expectation holds, "ice lost" warms more than "open water" and "ice in both" in autumn and winter, and narrows.

In [ ]:
#(t): What goes in: MODEL and EXPERIMENT's zones, and the mean and width changes behind them, with both edges
panels = fr.row_grid(
    [fr.Row(ice_zones_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS), zone_scale, 'Zone',
            edges=model_edges_da),
     fr.Row(mean_change_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS), tas_scales.change,
            f'Δ mean\n{EXPERIMENT} − hist-nat', edges=model_edges_da),
     fr.Row(width_change_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS), tas_scales.spread_change,
            f'Δ(Q95 − Q05)\n{EXPERIMENT} − hist-nat', edges=model_edges_da)],
    title=f'Where the ice changed, and the temperature: {MODEL}, {EXPERIMENT}, final {WINDOW} years',
)

In [ ]:
fig, axes = sip.zone_changes(tas_zone_means_ds['mean'], 'Δ mean (°C)', experiments=EXPERIMENTS, seasons=SEASONS,
                             title=f'Change in the mean temperature in each zone, final {WINDOW} years against hist-nat')

In [ ]:
fig, axes = sip.zone_changes(tas_zone_means_ds['width'], 'Δ(Q95 − Q05) (°C)', experiments=EXPERIMENTS, seasons=SEASONS,
                             title=f'Change in the width in each zone, final {WINDOW} years against hist-nat')

In [ ]:
fig, axes = sip.zone_changes(tas_zone_means_ds['lower tail'], 'Δ(Q50 − Q05) (°C)', experiments=EXPERIMENTS,
                             seasons=SEASONS,
                             title='Change in the lower (cold) tail in each zone: negative where the cold extremes '
                                   'warm more than the median')

In [ ]:
fig, axes = sip.zone_changes(tas_zone_means_ds['upper tail'], 'Δ(Q95 − Q50) (°C)', experiments=EXPERIMENTS,
                             seasons=SEASONS, title='Change in the upper (warm) tail in each zone')

### Across models and forcings

One point per model and forcing: the change in extent against the change in the mean, and in the width, averaged over the cells with ice in hist-nat. If losing ice drives them, more loss (further left) goes with more warming and more narrowing.

In [ ]:
panels = paper_panels.scatter_by_season(extent_change_ds['extent_change'], ice_region_change_ds['mean'],
                                        'Δ extent (million km²)', "Δ mean over hist-nat's ice (°C)",
                                        experiments=EXPERIMENTS, columns=SEASONS, style=fr.NOTEBOOK)

In [ ]:
panels = paper_panels.scatter_by_season(extent_change_ds['extent_change'], ice_region_change_ds['width'],
                                        'Δ extent (million km²)', "Δ(Q95 − Q05) over hist-nat's ice (°C)",
                                        experiments=EXPERIMENTS, columns=SEASONS, style=fr.NOTEBOOK)

### Year to year: where the sea ice and the temperature move together (JASMIN)

Strongly negative where a season with more ice is a colder season: that is where the ice sets the year-to-year variability of the temperature. ERA5's in the last row.

In [ ]:
#(t): The correlations, every model and ERA5
coupling_rows_da = xr.open_zarr(paths.find(paths.result('ice_tas_correlation', 'siconc', RUN)))['correlation'].load()

In [ ]:
#(t): Each one's own edge: hist-nat's final years, ERA5's 1979-2013
coupling_edges_da = xr.concat([ice_climatology_da.sel(experiment='hist-nat', drop=True),
                               era5_climatology_da.expand_dims(model=['ERA5'])], dim='model').expand_dims(edge=['edge'])

In [ ]:
panels = fr.map_grid(coupling_rows_da.sel(season=SEASONS), correlation_scale, row_dim='model', col_dim='season',
                     edges=coupling_edges_da.sel(season=SEASONS),
                     title='Year to year, how the sea ice and the temperature move together: hist-nat, and ERA5')